In [1]:
from tensorly_custom.tucker_tensor import validate_tucker_rank, tucker_normalize, TuckerTensor
from tensorly_custom.decomposition._tucker import tucker_to_tensor
from tensorly_custom.base import unfold
from tensorly_custom.tenalg import mode_dot
import tensorly_custom as tl
import pytensorlab as ptl
import numpy as np
from typing import List, Tuple
from utils import DATA_DIR, select_gpu, tree_to_device, notify_discord
from tucker_tensor import SparseTupleTensor
from typing import Optional, Union
import cupy as cp
import cupyx.scipy.sparse as cpx_sparse
import time
import torch
from cupyx.scipy import sparse
import os

device = select_gpu(1)
tl.set_backend("cupy")

# ----- cupy sparse methods -----

def fro_norm_coo(x):
    # x: cupyx.scipy.sparse.coo_matrix
    data = x.data
    return cp.sqrt((cp.abs(data) ** 2).sum())

def unfold_from_vectorized_sparse(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape,
    mode: int,
    to_dense: bool = False,
):
    """
    Unfold a sparse tensor that is stored as a vectorized CuPy sparse matrix.

    Parameters
    ----------
    vec_tensor : cupyx.scipy.sparse.spmatrix
        Sparse matrix of shape (np.prod(orig_shape), 1) created by
        `torch_sparse_to_cupy` for an N-D tensor.
    orig_shape : tuple[int, ...]
        Original N-D tensor shape, e.g. (I0, I1, I2).
    mode : int
        Mode along which to unfold.
    to_dense : bool, default False
        If True, return a dense cupy.ndarray.
        If False, return a cupy sparse COO matrix.

    Returns
    -------
    unfolded : cupy.ndarray or cupyx.scipy.sparse.coo_matrix
        Mode-`mode` unfolding of shape
        (orig_shape[mode], np.prod(orig_shape) // orig_shape[mode]).
    """
    # Make sure we're in COO format

    cu = vec_tensor.tocoo()

    row_cp = cu.row
    col_cp = cu.col
    data_cp = cu.data

    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    # ---- move to host and use int64 for safe arithmetic ----
    row_np = cp.asnumpy(row_cp).astype(np.int64)
    col_np = cp.asnumpy(col_cp).astype(np.int64)

    flat_np = row_np + col_np * np.int64(block_size)

    coords = np.unravel_index(flat_np, orig_shape)

    row_unf_np = coords[mode]

    other_coords = coords[:mode] + coords[mode + 1:]
    other_shape = tuple(s for i, s in enumerate(orig_shape) if i != mode)
    col_unf_np = np.ravel_multi_index(other_coords, other_shape)

    row_unf_cp = cp.asarray(row_unf_np)
    col_unf_cp = cp.asarray(col_unf_np)

    unfolded_shape = (orig_shape[mode], int(np.prod(other_shape)))
    unfolded = cpx_sparse.coo_matrix(
        (data_cp, (row_unf_cp, col_unf_cp)),
        shape=unfolded_shape,
    )

    if to_dense:
        return unfolded.toarray()
    return unfolded


def left_dense_mul_sparse(
    mat: cp.ndarray,
    sp: cpx_sparse.spmatrix
) -> Union[cp.ndarray, cpx_sparse.coo_matrix]:
    """
    Compute mat @ sp, choosing dense or sparse output based on a simple
    memory heuristic.

    mat: cupy ndarray of shape (R, I_mode)
    sp:  cupy sparse matrix of shape (I_mode, K)
    """
    sp = sp.tocoo()
    R, I_mode = mat.shape
    assert I_mode == sp.shape[0], f"mat shape {mat.shape} not compatible with sparse {sp.shape}"

    # Let CuPy handle dense @ sparse; result is cupy.ndarray
    return mat @ sp
def fold_unfolded_sparse_to_vec(
    unfolded: Union[cpx_sparse.spmatrix, cp.ndarray],
    old_shape: Tuple[int, ...],
    mode: int,
    new_dim: int,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Fold a mode-`mode` unfolded matrix back to a vectorized sparse tensor.

    unfolded:
        - sparse COO or any cupyx.scipy.sparse.spmatrix of shape (new_dim, prod(other_dims)), or
        - dense cupy.ndarray of the same shape.
    old_shape : original N-D shape BEFORE replacing dimension at `mode`
    mode      : mode index that was unfolded
    new_dim   : new size at `mode` (typically rank[mode])

    Returns
    -------
    vec_sparse : COO of shape (prod(new_shape), 1)
    new_shape  : tuple of ints, updated tensor shape
    """

    old_shape = tuple(old_shape)
    N = len(old_shape)

    new_shape = list(old_shape)
    new_shape[mode] = new_dim
    new_shape = tuple(new_shape)

    other_shape = tuple(s for i, s in enumerate(old_shape) if i != mode)

    if cpx_sparse.isspmatrix(unfolded):
        unfolded = unfolded.tocoo()
        row = unfolded.row
        col = unfolded.col
        data = unfolded.data
    else:
        row, col = cp.nonzero(unfolded)
        data = unfolded[row, col]

    coords_other = cp.unravel_index(col, other_shape)

    coords_full = []
    idx_other = 0
    for i in range(N):
        if i == mode:
            coords_full.append(row)
        else:
            coords_full.append(coords_other[idx_other])
            idx_other += 1

    coords_full = tuple(coords_full)

    size = int(np.prod(new_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    flat = cp.ravel_multi_index(coords_full, new_shape)

    # --- block encoding of flat indices ---
    row_vec = flat % block_size
    col_vec = flat // block_size

    n_blocks = int((size + block_size - 1) // block_size)
    vec_sparse = cpx_sparse.coo_matrix(
        (data, (row_vec, col_vec)),
        shape=(block_size, n_blocks),
    )
    vec_sparse.sum_duplicates()

    return vec_sparse, new_shape


def sparse_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    curr_shape: Tuple[int, ...],
    factor: cp.ndarray,
    mode: int,
    transpose_factor: bool = True,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Perform a mode-`mode` product on a vectorized sparse tensor (prod(curr_shape), 1),
    using a dense factor matrix, and return the new vectorized sparse tensor.

    vec_tensor: sparse COO (prod(curr_shape), 1)
    curr_shape: current tensor shape
    factor:     dense matrix of shape (I_mode, R_mode) (or R_mode, I_mode if transpose_factor=False)
    mode:       mode index in [0, len(curr_shape))
    transpose_factor: if True, use factor.T (for Tucker-style X ×_n W_n^T)

    Returns
    -------
    new_vec:   sparse COO (prod(new_shape), 1)
    new_shape: updated shape, with dimension at `mode` replaced by R_mode
    """
    curr_shape = tuple(curr_shape)
    I_mode = curr_shape[mode]

    # Factor handling
    if transpose_factor:
        # factor is (I_mode, R_mode) => mat is (R_mode, I_mode)
        assert factor.shape[0] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = tl.transpose(factor)  # (R_mode, I_mode)
    else:
        # factor is already (R_mode, I_mode)
        assert factor.shape[1] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = factor

    R_mode = mat.shape[0]

    # 1) Unfold current sparse tensor along this mode (sparse COO)
    unfolded = unfold_from_vectorized_sparse(
        vec_tensor,
        curr_shape,
        mode,
        to_dense=False,
    )  # shape: (I_mode, prod(other_dims))

    # 2) Left-multiply with dense matrix; currently returns dense cp.ndarray
    #    -> shape: (R_mode, prod(other_dims))
    unfolded_new = left_dense_mul_sparse(mat, unfolded)

    # 3) Fold back into a new vectorized sparse tensor with updated shape
    new_vec, new_shape = fold_unfolded_sparse_to_vec(
        unfolded_new,
        old_shape=curr_shape,
        mode=mode,
        new_dim=R_mode,
    )
    return new_vec, new_shape
def sparse_multi_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape: Tuple[int, ...],
    factors: List[cp.ndarray],
    modes: Optional[List[int]] = None,
    transpose_factors: bool = True,
) -> cp.ndarray:
    """
    multi_mode_dot for a vectorized sparse tensor (prod(orig_shape), 1),
    applying dense factor matrices along the given modes, **staying sparse**
    until the final (small) result, which is densified.

    vec_tensor: sparse COO (prod(orig_shape), 1)
    orig_shape: original tensor shape
    factors:    list of factor matrices, one per mode index
                factor[n] has shape (I_n, R_n)
    modes:      list of modes to apply; if None, uses range(len(factors))
    transpose_factors: if True, uses factors[n].T (Tucker-style)
    """
    if modes is None:
        modes = list(range(len(factors)))

    current_vec = vec_tensor
    current_shape = tuple(orig_shape)

    # Apply each mode in any order (commutes)
    for mode in modes:
        # print("Applying mode", mode)
        current_vec, current_shape = sparse_mode_dot_vec(
            current_vec,
            current_shape,
            factors[mode],
            mode=mode,
            transpose_factor=transpose_factors,
        )

    # At this point, current_vec is still sparse (prod(core_shape), 1)
    core_shape = current_shape  # typically your (50, 50, 50) or similar
    # should not overflow the cupy 32bit index limit if dimensions stay reasonable
    # Finally densify the small core
    coo = current_vec.tocoo()
    flat = coo.row
    data = coo.data

    # Build dense core
    coords = cp.unravel_index(flat, core_shape)
    core_dense = cp.zeros(core_shape, dtype=data.dtype)
    core_dense[coords] = data

    return core_dense

def print_elapsed_time(start_time, message=""):
    """Prints the elapsed time since start_time."""
    now = time.time()
    # if the message starts with indents (tabs), add the same number to the elapsed time print
    tabs = ""
    for char in message:
        if char == "\t":
            tabs += "\t"
        else:
            break
    elapsed_time = now - start_time
    minutes, seconds = divmod(elapsed_time, 60)
    if message:
        print(message)
    print(f"{tabs}Elapsed time: {int(minutes)} minutes and {seconds} seconds.")
    return now

def ptl_tucker_to_tensor(tucker: ptl.TuckerTensor,
                         skip_factor: Optional[int] = None) -> np.ndarray:
    """Reconstruct full tensor from Tucker representation, optionally skipping one factor."""
    factors = tucker.factors
    if skip_factor is not None:
        factors = [f for i, f in enumerate(factors) if i != skip_factor]
    return ptl.tmprod(tucker.core, factors, list(range(tucker.ndim)) if skip_factor is None else
                     [i for i in range(tucker.ndim) if i != skip_factor])


def sparse_dense_division_batched(X_csr_gpu, R_host, batch_rows=10_000):
    """
    X_csr_gpu : cupyx.scipy.sparse.csr_matrix on GPU
    R_host    : numpy.ndarray on CPU with same shape as X
    batch_rows: number of rows to process per batch
    """
    N, M = X_csr_gpu.shape
    assert R_host.shape == (N, M)

    # Store GPU sparse chunks and vstack at end
    batches = []
    counter = 0
    for start in range(0, N, batch_rows):
        stop = min(start + batch_rows, N)

        # Sparse slice is still on GPU (cheap)
        X_batch_gpu = X_csr_gpu[start:stop]

        # Move matching dense slice to GPU
        R_batch_gpu = cp.asarray(R_host[start:stop])

        # Elementwise division only for nonzeros (sparse × dense)
        # multiply is implemented efficiently on sparse
        Y_batch_gpu = X_batch_gpu.multiply(1.0 / R_batch_gpu)

        if counter < 1:
            # we print the sizes of the elements loaded into gpu
            print(f"Batch {counter}: X_batch_gpu shape: {X_batch_gpu.shape}, "
                  f"R_batch_gpu shape: {R_batch_gpu.shape}, "
                  f"Y_batch_gpu shape: {Y_batch_gpu.shape}, "
                  f"Y_batch_gpu nnz: {Y_batch_gpu.nnz}")
        counter += 1

        batches.append(Y_batch_gpu)

    # Combine back into one sparse matrix on GPU
    Y_gpu = sparse.vstack(batches, format='csr')
    return Y_gpu

dims = 2000
sparse_tensor = SparseTupleTensor.load_from_disk(
    dataset="fineweb_large",
    method="sc",
    dims=dims,
    map_location="cpu"
)
sparse_tensor.tensor_to_sparse("sparse")
ptl_tensor = ptl.SparseTensor(data=sparse_tensor.tensor.data,
                        indices=sparse_tensor.tensor.coords,
                        shape=sparse_tensor.tensor.shape)
sparse_tensor.tensor_to_sparse("torch")
sparse_tensor.tensor_to_sparse("cupy")
rank = [100, 100, 100]
tensor = sparse_tensor.tensor
shape = tuple(sparse_tensor.shape)
modes = list(range(len(rank)))
non_negative = True
init = "random"
random_state = 42

if init == "random":
    rng = tl.check_random_state(random_state)
    core = tl.tensor(
        rng.random_sample([rank[index] for index in range(len(modes))]) + 0.01,
        **tl.context(tensor),
    )
    factors = [
        tl.tensor(
            rng.random_sample((shape[mode], rank[index])), # we changed this to original shape
            **tl.context(tensor),
        )
        for index, mode in enumerate(modes)
    ]
else:
    (core, factors) = init

if non_negative is True:
    factors = [tl.abs(f) for f in factors]
    core = tl.abs(core)
else:
    raise NotImplementedError("Currently only non-negative=True is supported.")

2025-12-12 15:06:40.706310: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
mode_n = 0
skip_factor_tensors = [ptl.tens2mat(ptl_tensor, mode) for mode in modes]
X = skip_factor_tensors[mode_n]
core_np = tl.to_numpy(core)
factors_np = [tl.to_numpy(f) for f in factors]
ptl_tucker = ptl.TuckerTensor(core=core_np,
                            factors=factors_np)
Z = ptl_tucker_to_tensor(ptl_tucker, skip_factor=mode_n)
# Z = tucker_to_tensor((core, factors), skip_factor=mode_n)
# Z = unfold(Z, mode_n) # (50, 1000000)
Z = ptl.tens2mat(Z, mode_n)
A = factors[mode_n] # (1000, 50)
A_np = factors_np[mode_n]
# print("Z:", type(Z), Z.shape, "\nA:", type(A), A.shape)

# this breaks.
# R = tl.dot(A, Z) # (1000, 1000000)
R = A_np @ Z
X_csr_gpu = sparse.csr_matrix(X)

In [3]:
max_cuda_mem = torch.cuda.get_device_properties().total_memory
batch_rows = max_cuda_mem // (dims ** 2 * 12)
print(batch_rows)
frac = sparse_dense_division_batched(X_csr_gpu, R, batch_rows=batch_rows)

2108
Batch 0: X_batch_gpu shape: (1000, 1000000), R_batch_gpu shape: (1000, 1000000), Y_batch_gpu shape: (1000, 1000000), Y_batch_gpu nnz: 754107


In [3]:
max_cuda_mem = torch.cuda.get_device_properties().total_memory
print(f"Max CUDA memory: {max_cuda_mem}GB")

Max CUDA memory: 25297092608GB


In [21]:
batch_size = 235
dims * dims * rank[0] * batch_size

2115000000

In [26]:
25_297_092_608 == max_cuda_mem

True

In [32]:
2_115_000_000 == dims * dims * batch_size

True

In [38]:
dims * dims * batch_size * 12

25380000000

In [39]:
max_batch_size = max_cuda_mem // (dims * dims * 12)
print(f"Max batch size: {max_batch_size}")

Max batch size: 234


In [43]:
d = 2000
max_cuda_mem // (dims**2 *12)

527

In [3]:
check = False
epsilon = 1e-10
core_np = tl.to_numpy(core)
factors_np = [tl.to_numpy(f) for f in factors]
ptl_tucker = ptl.TuckerTensor(core=core_np,
                            factors=factors_np)
ptl_R = ptl_tucker_to_tensor(ptl_tucker)
if check:
    R = tucker_to_tensor((core, factors))
    R_np = tl.to_numpy(R)
    assert np.allclose(R_np, ptl_R), "R tensors from different implementations do not match!"

# we flatten to a sparse tensor, with values only where T has them

ptl_R = ptl_R.ravel().reshape(tensor.shape, order="F")
ptl_R = tl.clip(ptl_R, a_min=epsilon, a_max=None)
if check:
    R = R.ravel().reshape(tensor.shape, order="F")
    R = tl.clip(R, a_min=epsilon, a_max=None)
    R_np = tl.to_numpy(R)
    assert np.allclose(R_np, ptl_R), "R tensors from different implementations do not match!"

ValueError: cannot reshape array of size 8000000000 into shape (2147483647,4)

In [7]:
max_cuda_mem / 760000000

33.285648168421055

In [6]:
#print(batch_rows)
csr_tensor = tensor.tocsr()
ptl_X_R = sparse_dense_division_batched(csr_tensor, ptl_R, batch_rows=120000000)
if check:
    X_R = tensor / R
    print(f"X_R: type {type(X_R)}, shape {X_R.shape}")
    print(f"ptl_X_R: type {type(ptl_X_R)}, shape {ptl_X_R.shape}")
    assert cp.allclose(X_R.data, ptl_X_R.data)
# verify that both implementations give the same result


Batch 0: X_batch_gpu shape: (120000000, 1), R_batch_gpu shape: (120000000, 1), Y_batch_gpu shape: (120000000, 1), Y_batch_gpu nnz: 388633


In [1]:
(max_cuda_mem / 3)

NameError: name 'max_cuda_mem' is not defined

In [6]:
free_mem = torch.cuda.memory_reserved(0) - torch.cuda.memory_allocated(0)
print(f"Free CUDA memory: {free_mem} bytes")

Free CUDA memory: 0 bytes


In [5]:
br = 760_000_000
br = 8_432_364_202
csr_tensor = tensor.tocsr()
while True:
    print(br)
    ptl_X_R = sparse_dense_division_batched(csr_tensor, ptl_R, batch_rows=br)
    br += 10_000_000


8432364202
Batch 0: X_batch_gpu shape: (1000000000, 1), R_batch_gpu shape: (1000000000, 1), Y_batch_gpu shape: (1000000000, 1), Y_batch_gpu nnz: 754107
8442364202
Batch 0: X_batch_gpu shape: (1000000000, 1), R_batch_gpu shape: (1000000000, 1), Y_batch_gpu shape: (1000000000, 1), Y_batch_gpu nnz: 754107


OutOfMemoryError: Out of memory allocating 4,000,000,512 bytes (allocated so far: 24,048,434,176 bytes).

In [1]:
import multiprocessing
import os

In [3]:
multiprocessing.current_process()

<_MainProcess name='MainProcess' parent=None started>